In [3]:
import pandas as pd

icd_df = pd.read_excel("ICD_code_Assignment.xlsx")
cpt_df = pd.read_excel("cpt_code_assignment.xlsx")

icd_df = icd_df[['ICD Code', 'Description']]
icd_df.columns = ['Code', 'Description']

cpt_df = cpt_df[['CPT Code', 'Description']]
cpt_df.columns = ['Code', 'Description']


In [4]:
icd_df.head()
cpt_df.head()


,Code,Description
0,43200,Esophagoscopy Flexible Transoral Diagnostic
1,43202,Esophagoscopy Flexible Transoral With Biopsy
2,43220,Esophagoscopy Flex Balloon Dilat <30 Mm Diam
3,43235,Esophagogastroduodenoscopy Transoral Diagnostic
4,43236,Esophagogastroduodenoscopy Submucosal Injection


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

icd_embeddings = model.encode(icd_df["Description"].tolist())
cpt_embeddings = model.encode(cpt_df["Description"].tolist())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
def retrieve_codes(query, embeddings, df, top_k=3):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]
    return df.iloc[top_indices][['Code', 'Description']]


In [18]:
CLINICAL_TERMS = [
    "rectal bleeding", "internal hemorrhoids", "diverticulosis",
    "sessile polyp", "gastritis", "barrett", "colon polyps",
    "melanosis coli"
]

ANATOMY = [
    "rectum", "sigmoid colon", "cecum", "stomach",
    "duodenum", "esophagus", "anal verge"
]

PROCEDURES = [
    "colonoscopy", "egd", "biopsy", "polypectomy"
]

query_text = " ".join(CLINICAL_TERMS + PROCEDURES)
print(query_text)

report1 = """
Diagnosis:
Z86.0100 History of colon polyps
K64.8 Internal hemorrhoids
K57.90 Diverticulosis

Procedure:
Colonoscopy performed. Internal hemorrhoids were seen.
There was moderate sigmoid diverticulosis.
"""

report2 = """
Diagnosis:
Z12.11 Colon cancer screening

Procedure:
Colonoscopy performed. A 6 mm sessile polyp in descending colon
was removed using cold snare.
"""

report3 = """
Diagnosis:
K62.6 Ulcer of anus and rectum
K64.8 Other hemorrhoids
K62.5 Rectal bleeding

Procedure:
Colonoscopy with biopsy of rectal erosion.
"""

report4 = """
Diagnosis:
R07.89 Atypical chest pain
R10.11 Right upper quadrant pain
K29.70 Gastritis

Procedure:
EGD with biopsy. Findings suggestive of Barrett's esophagus.
"""



rectal bleeding internal hemorrhoids diverticulosis sessile polyp gastritis barrett colon polyps melanosis coli colonoscopy egd biopsy polypectomy


In [16]:
def extract_entities(text, keywords):
    return list(set([k for k in keywords if k in text]))

def preprocess(text):
    return text.lower().replace("\n", " ").strip()



In [19]:
text = preprocess(report1)

clinical_terms = extract_entities(text, CLINICAL_TERMS)
procedures = extract_entities(text, PROCEDURES)
anatomy = extract_entities(text, ANATOMY)

clinical_terms, procedures, anatomy


(['internal hemorrhoids', 'colon polyps', 'diverticulosis'],
 ['colonoscopy'],
 [])

In [20]:
query_text = " ".join(clinical_terms + procedures)

icd_results = retrieve_codes(query_text, icd_embeddings, icd_df)
cpt_results = retrieve_codes(query_text, cpt_embeddings, cpt_df)

icd_results, cpt_results


(       Code                                        Description
 133  K57.30  Diverticulosis of large intestine without perf...
 132  K57.10  Diverticulosis of small intestine without perf...
 152   K63.5                                     Polyp of colon,
      Code                                     Description
 18  45388  Colonoscopy Flx Ablation Tumor Polyp/Other Les
 13  45378      Colonoscopy Flx Dx W/Collj Spec When Pfrmd
 14  45380            Colonoscopy W/Biopsy Single/Multiple)

In [21]:
def process_report(report):
    text = preprocess(report)

    clinical_terms = extract_entities(text, CLINICAL_TERMS)
    anatomy = extract_entities(text, ANATOMY)
    procedures = extract_entities(text, PROCEDURES)

    query = " ".join(clinical_terms + procedures)

    icd = retrieve_codes(query, icd_embeddings, icd_df)
    cpt = retrieve_codes(query, cpt_embeddings, cpt_df)

    return {
        "Clinical Terms": clinical_terms,
        "Anatomical Locations": anatomy,
        "Diagnosis": clinical_terms,
        "Procedures": procedures,
        "ICD-10": icd["Code"].tolist(),
        "CPT": cpt["Code"].tolist(),
        "HCPCS": []
    }


In [22]:
outputs = [process_report(r) for r in [report1, report2, report3, report4]]
outputs


[{'Clinical Terms': ['internal hemorrhoids', 'colon polyps', 'diverticulosis'],
  'Anatomical Locations': [],
  'Diagnosis': ['internal hemorrhoids', 'colon polyps', 'diverticulosis'],
  'Procedures': ['colonoscopy'],
  'ICD-10': ['K57.30', 'K57.10', 'K63.5'],
  'CPT': [45388, 45378, 45380],
  'HCPCS': []},
 {'Clinical Terms': ['sessile polyp'],
  'Anatomical Locations': [],
  'Diagnosis': ['sessile polyp'],
  'Procedures': ['colonoscopy'],
  'ICD-10': ['K63.5', 'Z86.0102', 'Z86.0100'],
  'CPT': [45388, 45378, 45380],
  'HCPCS': []},
 {'Clinical Terms': ['rectal bleeding'],
  'Anatomical Locations': ['rectum'],
  'Diagnosis': ['rectal bleeding'],
  'Procedures': ['biopsy', 'colonoscopy'],
  'ICD-10': ['K51.911', 'K51.311', 'Z12.11'],
  'CPT': [45380, 45388, 45378],
  'HCPCS': []},
 {'Clinical Terms': ['barrett', 'gastritis'],
  'Anatomical Locations': ['esophagus'],
  'Diagnosis': ['barrett', 'gastritis'],
  'Procedures': ['biopsy', 'egd'],
  'ICD-10': ['K22.719', 'K22.70', 'K22.710'],

In [23]:
import json

with open("output.json", "w") as f:
    json.dump(outputs, f, indent=2)


In [25]:
!pip install pdfplumber
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.3 MB/s eta 0:00:00


In [26]:
pdf_text = extract_text_from_pdf("Input data for Assignment.pdf")
clean_text = preprocess(pdf_text)


In [41]:
def process_uploaded_pdf(pdf_path):
    raw_text = extract_text_from_pdf(pdf_path)
    text = preprocess(raw_text)

    entities = extract_entities_generic(text)
    diagnosis_query = " ".join(entities["diagnosis"])
    procedure_query = " ".join(entities["procedures"])


    icd = retrieve_codes(query, icd_embeddings, icd_df)
    cpt = retrieve_codes(query, cpt_embeddings, cpt_df)

    return {
        "Clinical Terms": entities,
        "ICD-10": icd["Code"].tolist(),
        "CPT": cpt["Code"].tolist()
    }


In [50]:
# Core
import re
import json
import pandas as pd
import numpy as np

# PDF parsing
!pip install pdfplumber
import pdfplumber

# Embeddings & similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [51]:
icd_df = pd.read_excel("ICD_code_Assignment.xlsx")
cpt_df = pd.read_excel("cpt_code_assignment.xlsx")

# Normalize column names
icd_df = icd_df[['ICD Code', 'Description']]
icd_df.columns = ['Code', 'Description']

cpt_df = cpt_df[['CPT Code', 'Description']]
cpt_df.columns = ['Code', 'Description']


In [52]:
model = SentenceTransformer("all-MiniLM-L6-v2")

icd_embeddings = model.encode(icd_df["Description"].tolist(), show_progress_bar=True)
cpt_embeddings = model.encode(cpt_df["Description"].tolist(), show_progress_bar=True)


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [53]:
def extract_text_from_pdf(pdf_path: str) -> str:
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


In [54]:
def preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


In [55]:
def extract_entities_generic(text: str):
    entities = {
        "diagnosis": set(),
        "procedures": set(),
        "anatomy": set(),
        "medications": set()
    }

    # --- ICD-like diagnosis phrases ---
    diagnosis_patterns = [
        r'history of [a-z ]+',
        r'internal hemorrhoids',
        r'diverticulosis',
        r'melanosis coli',
        r'gastritis',
        r'barrett\'s esophagus',
        r'no polyps found'
    ]

    for p in diagnosis_patterns:
        matches = re.findall(p, text)
        entities["diagnosis"].update(matches)

    # --- Procedures ---
    procedure_patterns = [
        'colonoscopy',
        'egd',
        'biopsy',
        'polypectomy',
        'retroflexion',
        'rectal exam',
        'monitored anesthesia care'
    ]

    for p in procedure_patterns:
        if p in text:
            entities["procedures"].add(p)

    # --- Anatomy ---
    anatomy_patterns = [
        'rectum',
        'sigmoid colon',
        'cecum',
        'proximal colon',
        'ileocecal valve',
        'appendiceal orifice'
    ]

    for a in anatomy_patterns:
        if a in text:
            entities["anatomy"].add(a)

    # --- Medications (for HCPCS) ---
    meds = ['propofol', 'lidocaine', 'lactated ringers']
    for m in meds:
        if m in text:
            entities["medications"].add(m)

    return entities


In [56]:
def retrieve_codes(query_text, embeddings, df, top_k=3):
    query_embedding = model.encode([query_text])
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]
    return df.iloc[top_indices]


In [60]:
def filter_icd_results(icd_df):
    """
    Filter ICD results to prefer diagnosis-related codes
    and avoid unrelated family-history codes.
    """
    return icd_df[
        icd_df["Code"].str.startswith(("Z86", "K", "R"))
    ]



def process_clinical_input(pdf_path=None, raw_text=None):
    if not pdf_path and not raw_text:
        raise ValueError("Provide pdf_path or raw_text")

    text = extract_text_from_pdf(pdf_path) if pdf_path else raw_text
    text = preprocess(text)

    entities = extract_entities_generic(text)

    diagnosis_query = " ".join(entities["diagnosis"])
    procedure_query = " ".join(entities["procedures"])

    icd_results = retrieve_codes(diagnosis_query, icd_embeddings, icd_df, top_k=5)
    icd_results = filter_icd_results(icd_results).head(3)

    cpt_results = retrieve_codes(procedure_query, cpt_embeddings, cpt_df, top_k=1)

    return {
        "Clinical Terms": list(entities["diagnosis"]),
        "Anatomical Locations": list(entities["anatomy"]),
        "Diagnosis": list(entities["diagnosis"]),
        "Procedures": list(entities["procedures"]),
        "ICD-10": icd_results["Code"].tolist(),
        "CPT": cpt_results["Code"].tolist(),
        "HCPCS": infer_hcpcs(entities["medications"])
    }


In [58]:
result = process_clinical_input(pdf_path="Input data for Assignment.pdf")
print(json.dumps(result, indent=2))


NameError: name 'filter_icd_results' is not defined

In [62]:
sample_text = """
Diagnosis:
Z86.0100 History of colon polyps
Z86.0100 - Personal history of colonic polyps K64.8 - Internal hemorrhoids K57.90 - Diverticulosis
Procedure:
Procedure Code Colonoscopy
Anesthesia Type : Monitored Anesthesia Care ASA Class : II
Lactated Ringers - Solution, Intravenous as directed - 350 00
, Last Administered By:
Smith, George At 1041 on 07/07/2025 Lidocaine HCI 2% Solution, IV - 20 00 , Last Administered By: Smith, George At 1023 on 07/07/2025 Propofol 500 MG/50ML Emulsion, Intravenous - 240 00 , Last Administered By: Smith, George At 1041 on 07/07/2025
Colonoscopy PROCEDURE: There was nothing precluding endoscopy on history or physical exam. Informed consent was obtained with risks and benefits explained to the patient. The patient tolerated the procedure well. There no immediate complications.
The patient was placed in left lateral decubitus position. A rectal exam was performed.
The pediatric colonoscope was inserted into the rectum and carefully advanced to the cecum. The cecum was identified by the ileocecal valve, the triradiate fold and appendiceal orifice. Careful inspection was made as the colonoscope was removed including retroflexion in the rectum. Findings- The preparation was good. There was melanosis coli in the proximal colon. There was moderate sigmoid diverticulosis.
Internal hemorrhoids were seen. IMPRESSION: The patient is an 82-year-old female with history of colon polyps. Today's exam did not reveal any polyps. she did have melanosis coli, diverticulosis and internal hemorrhoids. PLAN: No routine colonoscopy
Colonoscopy The patient tolerated the procedure without complications The colonoscopy was uneventful
"""

result = process_clinical_input(raw_text=sample_text)
print(json.dumps(result, indent=2))


{
  "Clinical Terms": [
    "history of colonic polyps k",
    "internal hemorrhoids",
    "history of colon polyps",
    "diverticulosis",
    "history of colon polyps z",
    "melanosis coli"
  ],
  "Anatomical Locations": [
    "appendiceal orifice",
    "cecum",
    "ileocecal valve",
    "proximal colon",
    "rectum"
  ],
  "Diagnosis": [
    "history of colonic polyps k",
    "internal hemorrhoids",
    "history of colon polyps",
    "diverticulosis",
    "history of colon polyps z",
    "melanosis coli"
  ],
  "Procedures": [
    "monitored anesthesia care",
    "retroflexion",
    "colonoscopy",
    "rectal exam"
  ],
  "ICD-10": [
    "Z86.0100",
    "Z86.0102",
    "Z86.0101"
  ],
  "CPT": [
    45380
  ],
  "HCPCS": [
    "J3490",
    "A4216"
  ]
}


In [43]:
def infer_hcpcs(medications):
    hcpcs = []
    if 'propofol' in medications or 'lidocaine' in medications:
        hcpcs.extend(["J3490", "A4216"])
    return hcpcs


In [38]:
def categorize_entities(entities):
    clinical_terms = []
    anatomy = []
    procedures = []

    for e in entities:
        if e in ["colon", "sigmoid", "rectum", "cecum"]:
            anatomy.append(e)
        elif e in ["colonoscopy", "retroflexion", "endoscopy"]:
            procedures.append(e)
        else:
            clinical_terms.append(e)

    return clinical_terms, anatomy, procedures


NameError: name 'entities' is not defined

In [44]:
final_output = {
    "Clinical Terms": list(entities["diagnosis"]),
    "Anatomical Locations": list(entities["anatomy"]),
    "Diagnosis": list(entities["diagnosis"]),
    "Procedures": list(entities["procedures"]),
    "ICD-10": icd_results["Code"].tolist(),
    "CPT": cpt_results["Code"].tolist(),
    "HCPCS": infer_hcpcs(entities["medications"])
}


NameError: name 'entities' is not defined

In [65]:
# =========================
# 1. IMPORTS
# =========================
!pip install pdfplumber
import re
import json
import pandas as pd
import pdfplumber
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity



# =========================
# 2. LOAD ICD & CPT DATA
# =========================
icd_df = pd.read_excel("ICD_code_Assignment.xlsx")
cpt_df = pd.read_excel("cpt_code_assignment.xlsx")

icd_df = icd_df[['ICD Code', 'Description']]
icd_df.columns = ['Code', 'Description']

cpt_df = cpt_df[['CPT Code', 'Description']]
cpt_df.columns = ['Code', 'Description']


# =========================
# 3. EMBEDDINGS (RAG)
# =========================
model = SentenceTransformer("all-MiniLM-L6-v2")
icd_embeddings = model.encode(icd_df["Description"].tolist())
cpt_embeddings = model.encode(cpt_df["Description"].tolist())


def retrieve_codes(query, embeddings, df, top_k=5):
    q_emb = model.encode([query])
    scores = cosine_similarity(q_emb, embeddings)[0]
    idx = scores.argsort()[-top_k:][::-1]
    return df.iloc[idx]


# =========================
# 4. PDF / TEXT INGESTION
# =========================
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            if page.extract_text():
                text += page.extract_text() + "\n"
    return text


def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# =========================
# 5. CLINICAL NORMALIZATION
# =========================
ANATOMY_LIST = [
    "anal canal", "rectum", "sigmoid colon", "descending colon",
    "splenic flexure", "transverse colon", "hepatic flexure",
    "ascending colon", "cecum", "terminal ileum"
]


def extract_anatomy(text):
    return [a.title() for a in ANATOMY_LIST if a in text]


def extract_explicit_icds(text):
    return list(set(re.findall(r'\b[A-Z]\d{2}\.\d{1,2}\b', text)))


def infer_clinical_terms(text):
    terms = []

    if "colon cancer screening" in text or "z12.11" in text:
        terms.append("Colon cancer screening")

    if "hemorrhoids" in text:
        terms.append("Hemorrhoids")

    if "sessile polyp" in text:
        terms.append("Sessile polyp")

    if "cold snare" in text:
        terms.append("Polyp removal")

    if "no immediate complication" in text:
        terms.append("No immediate complication")

    return terms


def infer_procedures(text):
    procs = []
    if "colonoscopy" in text:
        procs.append("Colonoscopy")
    if "cold snare" in text:
        procs.append("Cold snare polypectomy")
    if "biopsy" in text and "cold snare" not in text:
        procs.append("Biopsy")
    return procs


def infer_cpt(text):
    if "cold snare" in text:
        return ["45385"]
    if "biopsy" in text:
        return ["45380"]
    if "colonoscopy" in text:
        return ["45378"]
    return []


def infer_hcpcs(text):
    if "propofol" in text or "lidocaine" in text:
        return ["J3490", "A4216"]
    return []


# =========================
# 6. MAIN PIPELINE
# =========================
def process_clinical_input(pdf_path=None, raw_text=None):
    if not pdf_path and not raw_text:
        raise ValueError("Provide pdf_path or raw_text")

    text = extract_text_from_pdf(pdf_path) if pdf_path else raw_text
    text = preprocess(text)

    clinical_terms = infer_clinical_terms(text)
    anatomy = extract_anatomy(text)
    diagnosis = clinical_terms.copy()
    procedures = infer_procedures(text)

    explicit_icds = extract_explicit_icds(text)

    if explicit_icds:
        icd_codes = explicit_icds
    else:
        icd_candidates = retrieve_codes(
            " ".join(clinical_terms), icd_embeddings, icd_df, top_k=5
        )
        icd_codes = icd_candidates["Code"].tolist()

    cpt_codes = infer_cpt(text)
    hcpcs_codes = infer_hcpcs(text)

    return {
        "Clinical Terms": clinical_terms,
        "Anatomical Locations": anatomy,
        "Diagnosis": diagnosis,
        "Procedures": procedures,
        "ICD-10": icd_codes,
        "CPT": cpt_codes,
        "HCPCS": hcpcs_codes
    }


# =========================
# 7. EXAMPLE USAGE
# =========================

# --- PDF input ---
# result = process_clinical_input(pdf_path="Input data for Assignment.pdf")

# --- OR text input ---
sample_text = """
Diagnosis:
Pre-operative Diagnosis
R07.89 - Atypical chest pain R10.11 - Right upper quadrant abdominal pain
Post-operative Diagnosis
R07.89 - Atypical chest pain R10.11 - RUQ pain K29.70 - Gastritis
Procedures:
Procedure Code EGD w/Biopsy
Anesthesia Type: Monitored Anesthesia Care
Lactated Ringers - Solution, Intravenous as directed - 200 00, Last Administered By:
Smith, George At 1419 on 07/07/2025 Lidocaine HCI 2% Solution, IV - 60 00 , Last Administered By: Smith, George At 1408 on 07/07/2025 Propofol 500 MG/50ML Emulsion, Intravenous - 350 00, Last Administered By: Smith, George At 1419 on 07/07/2025
EGD PROCEDURE: There was nothing precluding endoscopy on history or physical exam. Informed consent was obtained with risks and benefits explained to the patient.
The patient tolerated the procedure well. There were no immediate complications The patient was placed in the left lateral decubitus position. The Olympus endoscope was inserted into the esophagus under direct visualization. It was advanced through the esophagus, into the stomach and through the pylorus to the duodenal bulb and 2nd portion of the duodenum. Careful inspection was made as the endoscope was removed including retroflexion in the stomach. Findings- In the distal esophagus there was an irregular Z-line from 38-39 cm suggestive of Barrett's esophagus. This was examined both white light and narrow band imaging. Biopsies were obtained of the distal esophagus. In the stomach there was mild antral and body gastritis. Biopsies were obtained for H.pylori. There were no ulcers or masses seen. The visualized portion of the duodenal appeared normal without ulcers or inflammation. IMPRESSION: The patient is 55-year-old male with atypical chest pain and right upper quadrant pain. Biopsies today were obtained for H.pylori and Barrett's esophagus. If the patient does Barrett's esophagus and a repeat EGD with biopsy in 6 months will be recommended.
Post Operative Impression
EGD The patient tolerated the procedure without complications. The EGD was uneventful.
"""

result = process_clinical_input(raw_text=sample_text)

print(json.dumps(result, indent=2))


{
  "Clinical Terms": [
    "No immediate complication"
  ],
  "Anatomical Locations": [],
  "Diagnosis": [
    "No immediate complication"
  ],
  "Procedures": [
    "Biopsy"
  ],
  "ICD-10": [
    "R15.0",
    "K56.609",
    "Z53.8",
    "K56.699",
    "T17.908D"
  ],
  "CPT": [
    "45380"
  ],
  "HCPCS": [
    "J3490",
    "A4216"
  ]
}
